# 11 · ConceptNet 100k · sweep `k_neighbors = 2…5` → Wishart → Graphex → Haken

**Часть 2 из 2.** Этот notebook сравнивает четыре значения числа соседей Wishart: **2, 3, 4, 5**. Он использует один и тот же GloVe-кNN, рассчитанный ноутбуком 10, и для каждого `k` берёт первые `k` уже отсортированных соседей. Поэтому дорогостоящий GPU-поиск **не повторяется четыре раза** и все варианты используют одну и ту же геометрию GloVe.

Дальше четыре независимых пайплайна `Wishart → finite Graphex W/S/I/R → exact decode → Haken` выполняются **параллельно отдельными процессами**. На обычной Colab RAM notebook запускает два процесса одновременно; на High-RAM — до четырёх. Drive используется только для контрольных точек и результатов, вычисления выполняются на `/content`.

**Научная оговорка:** `k` меняет локальную плотность, используемую Wishart, поэтому сравнивать варианты надо не только по номеру `k`, но и по фактическому числу кластеров/шуму, степени укрупнения, полному размеру архива и искажению медленного спектрального подпространства. Точный decoder проверяется для каждого `k` отдельно.

In [ ]:
# 1. Конфигурация sweep. Обычно менять нужно только RUN_SWEEP и при необходимости BASE_RUN.
BRANCH = "feature/glove-wishart-graphex-cn100k-lossless-l4"
DRIVE_ROOT = "/content/drive/MyDrive/SemanticMap/colab/wishart"
SOURCE_RUN = "wishart-dictionary-100k-gpu-gpu"       # N=100000, источник level_000
BASE_RUN = "wishart-glove-graphex-cn100k-01"         # результат notebook 10; его kNN должен иметь k>=5
RUN_SWEEP = "wishart-glove-graphex-cn100k-k2-5-sweep-01"
GLOVE_REL = "data/glove.6B.100d.txt"
SCRATCH_ROOT = "/content/semmap-glove-graphex-100k"
EXPECTED_NODES = 100000
K_VALUES = (2, 3, 4, 5)
QUERY_BATCH = 256
MAX_FIGURES = 4096
MIN_SUPPORT = 5
PARALLEL_RUNS = 0       # 0 = auto: 4 при >=44 GiB доступной RAM, иначе 2
MIN_PARALLEL_RAM_GIB = 18


In [ ]:
# 2. Preflight. В notebook 11 GPU не обязателен: kNN уже рассчитан notebook 10.
import os, sys, shutil, platform
from pathlib import Path
try:
    import psutil
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil>=5.9,<8"], check=True)
    import psutil

available_gib = psutil.virtual_memory().available / 2**30
cpu_count = os.cpu_count() or 1
auto_workers = 4 if available_gib >= 44 else 2
WORKERS = PARALLEL_RUNS or auto_workers
WORKERS = min(WORKERS, len(K_VALUES), max(1, cpu_count // 2))
print({"python": platform.python_version(), "cpus": cpu_count,
       "available_ram_gib": round(available_gib, 1),
       "scratch_free_gib": round(shutil.disk_usage('/content').free / 2**30, 1),
       "parallel_runs": WORKERS})
if WORKERS < 2 or available_gib < MIN_PARALLEL_RAM_GIB:
    raise RuntimeError("Для параллельного k-sweep выберите Colab с >=18 GiB доступной RAM (High-RAM рекомендуется).")
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not required in notebook 11")
except Exception:
    print("GPU: not required in notebook 11")


In [ ]:
# 3. Код из Git: тяжёлая логика остаётся в semgraphex, notebook — только orchestration.
import subprocess, json, hashlib, time, csv, statistics
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"  # важно: несколько процессов, без nested BLAS oversubscription
ROOT = Path(SCRATCH_ROOT); ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/SemanticMap/semgraphex.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH, "--depth", "1"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))
print("semgraphex commit:", COMMIT)


In [ ]:
# 4. Drive + исходный 100k-граф + только нужные общие артефакты notebook 10.
from google.colab import drive
drive.mount("/content/drive")
from scipy import sparse
from semmap_haken.wishart_colab import stage_from_drive, sync_tree

DRIVE = Path(DRIVE_ROOT)
SOURCE = DRIVE / "runs" / SOURCE_RUN / "level_000"
BASE_REMOTE = DRIVE / "runs" / BASE_RUN
BASE_LOCAL = ROOT / "base" / BASE_RUN
SWEEP_REMOTE = DRIVE / "runs" / RUN_SWEEP
SWEEP_LOCAL = ROOT / "runs" / RUN_SWEEP
LOCAL_INPUT = ROOT / "inputs" / "level_000"
GLOVE_REMOTE = DRIVE / GLOVE_REL

for required in (SOURCE / "adjacency.npz", SOURCE / "membership.json", SOURCE / "relations" / "index.json"):
    if not required.is_file(): raise FileNotFoundError(required)
if not (DRIVE / "runs" / SOURCE_RUN / "COMPLETED").is_file():
    raise RuntimeError("Исходный 100k Wishart-run должен иметь COMPLETED")
if not LOCAL_INPUT.exists(): shutil.copytree(SOURCE, LOCAL_INPUT)
shape = sparse.load_npz(LOCAL_INPUT / "adjacency.npz").shape
if shape != (EXPECTED_NODES, EXPECTED_NODES): raise RuntimeError(f"Ожидалось 100000 узлов, получено {shape}")

# Staging только embedding + kNN, без старых codec/haken результатов.
for rel in ("embedding", "knn"):
    src = BASE_REMOTE / rel
    if not src.is_dir(): raise FileNotFoundError(f"Сначала выполните notebook 10: нет {src}")
    stage_from_drive(src, BASE_LOCAL / rel, compare_mode="sha256")
if (BASE_REMOTE / "input.json").is_file():
    stage_from_drive(BASE_REMOTE / "input.json", BASE_LOCAL / "input.json", compare_mode="sha256")
for phase in ("align", "knn"):
    checkpoint = BASE_REMOTE / "checkpoints" / f"{phase}.json"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Notebook 10 не опубликовал checkpoint {phase}: {checkpoint}")
    stage_from_drive(checkpoint, BASE_LOCAL / "checkpoints" / f"{phase}.json", compare_mode="sha256")

GLOVE = ROOT / "inputs" / "glove.6B.100d.txt"
GLOVE.parent.mkdir(parents=True, exist_ok=True)
if not GLOVE_REMOTE.is_file(): raise FileNotFoundError(GLOVE_REMOTE)
if not GLOVE.is_file() or GLOVE.stat().st_size != GLOVE_REMOTE.stat().st_size:
    shutil.copy2(GLOVE_REMOTE, GLOVE)
SWEEP_LOCAL.mkdir(parents=True, exist_ok=True); SWEEP_REMOTE.mkdir(parents=True, exist_ok=True)
print("source shape:", shape, "base kNN:", BASE_REMOTE, "sweep:", SWEEP_REMOTE)


In [ ]:
# 5. Integrity: общий kNN обязан быть отсортирован и содержать >= max(K_VALUES) соседей.
import numpy as np
from semmap_haken.glove_graphex_100k import sha, write_json, run_identity, stage_commit, stage_valid

base_knn_path = BASE_LOCAL / "knn" / "knn.npz"
base_valid_path = BASE_LOCAL / "knn" / "valid.npy"
base_report_path = BASE_LOCAL / "knn" / "report.json"
for p in (base_knn_path, base_valid_path, base_report_path,
          BASE_LOCAL / "embedding" / "vectors.npy",
          BASE_LOCAL / "embedding" / "coverage.npy",
          BASE_LOCAL / "embedding" / "report.json"):
    if not p.is_file(): raise FileNotFoundError(p)
base_report = json.loads(base_report_path.read_text(encoding="utf8"))
if not stage_valid(BASE_LOCAL, "align") or not stage_valid(BASE_LOCAL, "knn"):
    raise RuntimeError("Базовые embedding/kNN не прошли проверку checkpoints notebook 10")
with np.load(base_knn_path) as data:
    BASE_INDICES = data["indices"].astype(np.int32, copy=False)
    BASE_DISTANCES = data["distances"].astype(np.float32, copy=False)
VALID = np.load(base_valid_path)
if BASE_INDICES.shape != BASE_DISTANCES.shape or BASE_INDICES.shape[0] != len(VALID):
    raise RuntimeError("Повреждён общий kNN")
if BASE_INDICES.shape[1] < max(K_VALUES):
    raise RuntimeError(f"Notebook 10 должен построить хотя бы k={max(K_VALUES)}, найдено {BASE_INDICES.shape[1]}")
if np.any(BASE_DISTANCES[:, 1:] + 1e-7 < BASE_DISTANCES[:, :-1]):
    raise RuntimeError("Соседи в базовом kNN не отсортированы по расстоянию; slicing k некорректен")
BASE_KNN_SHA = sha(base_knn_path)
print({"base_k": BASE_INDICES.shape[1], "queries": len(VALID), "knn_sha256": BASE_KNN_SHA})


In [ ]:
# 6. Материализуем четыре независимых run-а. embedding локально hard-link'ится, kNN режется по первым k.
from types import SimpleNamespace

def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists(): return
    try: os.link(src, dst)
    except OSError: shutil.copy2(src, dst)

children = {}
for k in K_VALUES:
    local = SWEEP_LOCAL / f"k_{k:02d}"
    remote = SWEEP_REMOTE / f"k_{k:02d}"
    # Resume существующего дочернего run с Drive.
    if (remote / "input.json").is_file():
        stage_from_drive(remote, local, compare_mode="sha256")
    local.mkdir(parents=True, exist_ok=True); remote.mkdir(parents=True, exist_ok=True)

    args = SimpleNamespace(expected_nodes=EXPECTED_NODES, k=k, batch=QUERY_BATCH,
                           max_figures=MAX_FIGURES, min_support=MIN_SUPPORT)
    identity = run_identity(LOCAL_INPUT, GLOVE, args)
    input_path = local / "input.json"
    if input_path.is_file():
        if json.loads(input_path.read_text(encoding="utf8")) != identity:
            raise RuntimeError(f"k={k}: config/code/source изменились; задайте новый RUN_SWEEP")
    else:
        write_json(input_path, identity)

    # Align-artifacts общие и неизменные; hardlink не дублирует 40MB vectors.npy на /content.
    for name in ("vectors.npy", "coverage.npy", "report.json"):
        link_or_copy(BASE_LOCAL / "embedding" / name, local / "embedding" / name)
    if not stage_valid(local, "align"): stage_commit(local, "align")

    if not stage_valid(local, "knn"):
        (local / "knn").mkdir(parents=True, exist_ok=True)
        link_or_copy(base_valid_path, local / "knn" / "valid.npy")
        np.savez_compressed(local / "knn" / "knn.npz",
                            indices=BASE_INDICES[:, :k], distances=BASE_DISTANCES[:, :k])
        write_json(local / "knn" / "report.json", {
            "backend": "derived-prefix-of-common-sorted-knn",
            "k": k, "queries": len(VALID), "source_k": int(BASE_INDICES.shape[1]),
            "source_knn_sha256": BASE_KNN_SHA, "source_run": BASE_RUN,
            "note": "same GloVe geometry for every k; no repeated GPU neighbor search"
        })
        stage_commit(local, "knn")
    write_json(local / "sweep_lineage.json", {
        "sweep": RUN_SWEEP, "k": k, "k_values": list(K_VALUES),
        "source_run": BASE_RUN, "source_knn_sha256": BASE_KNN_SHA,
        "code_revision": COMMIT
    })
    # Initial checkpoint publish; COMPLETED is never copied here.
    sync_tree(local, remote, compare_mode="size_mtime", final=False)
    children[k] = (local, remote)

write_json(SWEEP_LOCAL / "sweep_manifest.json", {
    "run": RUN_SWEEP, "k_values": list(K_VALUES), "parallel_runs": WORKERS,
    "source_run": SOURCE_RUN, "base_knn_run": BASE_RUN,
    "base_knn_sha256": BASE_KNN_SHA, "code_revision": COMMIT,
    "expected_nodes": EXPECTED_NODES, "max_figures": MAX_FIGURES,
    "min_support": MIN_SUPPORT
})
sync_tree(SWEEP_LOCAL / "sweep_manifest.json", SWEEP_REMOTE / "sweep_manifest.json",
          compare_mode="sha256", final=False)
print("Prepared:", {k: str(v[0]) for k, v in children.items()})


In [ ]:
# 7. Параллельный запуск. Каждый k идёт в отдельном subprocess; Drive-sync сериализован lock'ом.
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

ENV = dict(os.environ, PYTHONPATH=str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""))
SYNC_LOCK = Lock()

def command_for(k, local):
    return [sys.executable, "-m", "semmap_haken.glove_graphex_100k",
            "--source-level", str(LOCAL_INPUT), "--glove", str(GLOVE),
            "--run", str(local), "--expected-nodes", str(EXPECTED_NODES),
            "--k", str(k), "--batch", str(QUERY_BATCH),
            "--max-figures", str(MAX_FIGURES), "--min-support", str(MIN_SUPPORT)]

def run_one(k):
    local, remote = children[k]
    if (local / "COMPLETED").is_file() and (local / "verification.json").is_file():
        return {"k": k, "status": "resume-completed", "seconds": 0.0}
    started = time.perf_counter()
    base = command_for(k, local)
    for stage in ("wishart", "codec", "haken", "verify"):
        proc = subprocess.run(base + ["--stage", stage], cwd=REPO, env=ENV,
                              text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if proc.returncode:
            raise RuntimeError(f"k={k}, stage={stage} failed:\n{proc.stdout[-5000:]}")
        print(f"k={k}: {stage} OK", flush=True)
        # Не запускаем конкурирующие копирования больших ZIP на Drive.
        with SYNC_LOCK:
            sync_tree(local, remote, compare_mode="size_mtime", final=(stage == "verify"))
    return {"k": k, "status": "completed", "seconds": time.perf_counter() - started}

results = []
with ThreadPoolExecutor(max_workers=WORKERS, thread_name_prefix="k-sweep") as pool:
    futures = {pool.submit(run_one, k): k for k in K_VALUES}
    for future in as_completed(futures):
        k = futures[future]
        try:
            item = future.result(); results.append(item); print("DONE", item, flush=True)
        except Exception as exc:
            print(f"FAILED k={k}: {exc}", flush=True)
            raise
results.sort(key=lambda x: x["k"])
print(json.dumps(results, indent=2))


In [ ]:
# 8. Единая сравнительная таблица: compression / archive / Wishart / Haken.
rows = []
for k in K_VALUES:
    local, remote = children[k]
    if not (local / "COMPLETED").is_file(): raise RuntimeError(f"k={k} не завершён")
    wishart = json.loads((local / "knn" / "wishart.json").read_text(encoding="utf8"))
    codec = json.loads((local / "codec_report.json").read_text(encoding="utf8"))
    haken = json.loads((local / "dynamics" / "haken.json").read_text(encoding="utf8"))
    verify = json.loads((local / "verification.json").read_text(encoding="utf8"))
    angles = haken["nontrivial_principal_angles_degrees"]
    selected = min(codec["models"].values(), key=lambda x: x["archive_bytes"])
    rows.append({
        "k": k,
        "clusters": wishart["clusters"],
        "noise_and_oov": wishart["noise_and_oov"],
        "quotient_nodes": haken["quotient_nodes"],
        "compression_fraction": 1 - haken["quotient_nodes"] / EXPECTED_NODES,
        "archive_bytes": codec["selected_archive_bytes"],
        "baseline_bytes": codec["baseline_bytes"],
        "archive_ratio": codec["ratio"],
        "net_saved_bytes": codec["net_saved_bytes"],
        "lossless": bool(verify["lossless"]),
        "W_records": selected["W"], "S_records": selected["S"],
        "I_records": selected["I"], "R_records": selected["R"],
        "shapes": selected["shapes"], "occurrences": selected["occurrences"],
        "stationary_mass_TV": haken["stationary_mass_total_variation"],
        "principal_angle_mean_deg": float(np.mean(angles)) if angles else None,
        "principal_angle_max_deg": float(np.max(angles)) if angles else None,
    })

# Одинаковый baseline — обязательная comparability-проверка.
if len({r["baseline_bytes"] for r in rows}) != 1:
    raise RuntimeError("Baseline bytes различаются между k: сравнение некорректно")
if not all(r["lossless"] for r in rows):
    raise RuntimeError("Хотя бы один k не прошёл lossless verification")

write_json(SWEEP_LOCAL / "sweep_summary.json", {"rows": rows})
with (SWEEP_LOCAL / "sweep_summary.csv").open("w", newline="", encoding="utf8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    print(json.dumps(rows, indent=2))


In [ ]:
# 9. Две диагностические кривые: реальное сжатие и сохранение медленных мод.
import matplotlib.pyplot as plt
ks = [r["k"] for r in rows]
fig = plt.figure(figsize=(7, 4))
plt.plot(ks, [r["archive_ratio"] for r in rows], marker="o")
plt.axhline(1.0, linestyle="--")
plt.xlabel("k_neighbors")
plt.ylabel("archive / baseline")
plt.title("Physical archive ratio vs k")
plt.grid(alpha=.25)
plt.show()

fig = plt.figure(figsize=(7, 4))
plt.plot(ks, [r["principal_angle_max_deg"] for r in rows], marker="o")
plt.xlabel("k_neighbors")
plt.ylabel("max principal angle, degrees")
plt.title("Slow-subspace distortion vs k")
plt.grid(alpha=.25)
plt.show()


In [ ]:
# 10. Финальная публикация sweep: summary сначала, COMPLETED — последней записью.
sync_tree(SWEEP_LOCAL / "sweep_summary.json", SWEEP_REMOTE / "sweep_summary.json",
          compare_mode="sha256", final=False)
sync_tree(SWEEP_LOCAL / "sweep_summary.csv", SWEEP_REMOTE / "sweep_summary.csv",
          compare_mode="sha256", final=False)
for k in K_VALUES:
    if not (SWEEP_REMOTE / f"k_{k:02d}" / "COMPLETED").is_file():
        raise RuntimeError(f"Drive run k={k} не опубликован как COMPLETED")
(SWEEP_LOCAL / "COMPLETED").write_text("all k=2..5 validated\n", encoding="ascii")
sync_tree(SWEEP_LOCAL, SWEEP_REMOTE, compare_mode="sha256", final=True)
print("CHECKPOINT published")
print("CHECKPOINT summary")
print("ГОТОВО", SWEEP_REMOTE)


## Как интерпретировать sweep

Не выбирайте `k` только по минимальному размеру архива. Для каждого варианта одновременно смотрите: (1) число кластеров и шум Wishart; (2) фактическую степень укрупнения; (3) **полный** `archive_ratio`, включая residual; (4) долю W против R; (5) principal angles медленного подпространства и stationary-mass TV. Если меньший архив достигается ценой резкого роста spectral distortion, это компромисс, а не автоматически лучший `k`.

Каждый дочерний каталог `k_02 … k_05` имеет собственные `input.json`, checkpoints, `graph_exact.zip`, `verification.json`, `dynamics/haken.json` и `COMPLETED`. Общий `sweep_summary.{json,csv}` создаётся только после успешного завершения всех четырёх вариантов.